# Magellan Spectroscopic Follow-Up Planner

This notebook generates an observing catalog for Magellan spectroscopic follow-up
of SN Ia candidates discovered by the RubinAlerts pipeline.

## Workflow

1. Load peak-fitted SN Ia candidates from the SQLite cache (written by `supernova_monitor.ipynb`)
2. Recompute merit scores for the planned observing night
3. Filter for targets observable from Las Campanas
4. Sort by RA and write the Magellan TCS catalog file

## Prerequisites

Run `supernova_monitor.ipynb` through the "Cache peak-fitted candidates" cell first.
That populates the `peak_fit_targets` table in the SQLite cache.

In [ ]:
# ========================= MAGELLAN PLANNING PARAMETERS =========================
CACHE_DIR          = './cache/data'    # Must match supernova_monitor.ipynb
OBS_DATE           = '2026-03-15'      # UT date of the observing night (YYYY-MM-DD)
INSTRUMENT         = 'LDSS3C'          # Magellan instrument (see presets below)
MAX_AIRMASS        = 2.0               # Maximum airmass cutoff
MIN_HOURS_UP       = 1.0               # Minimum hours above airmass limit
MIN_MERIT          = 0.01              # Minimum merit score to include
INCLUDE_NO_FIT     = False             # Include targets without successful peak fits?
OUTPUT_DIR         = '.'               # Output directory for catalog file

# Merit function tuning
MERIT_TAU          = 10.0              # Time decay half-width (days)
MERIT_MAG_OPTIMAL  = 20.5              # Optimal peak magnitude for Magellan
MERIT_MAG_BRIGHT   = 18.0              # Bright end of useful range
MERIT_MAG_FAINT    = 23.0              # Faint end of useful range
# ================================================================================

# Available instrument presets:
#   LDSS3C, IMACS-f2, IMACS-f4, MagE, FIRE-echelle, FIRE-longslit

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('magellan_planning.ipynb'))))

import warnings; warnings.filterwarnings('ignore')
import logging
import pandas as pd
import numpy as np
from datetime import datetime
from IPython.display import display, HTML
from astropy.time import Time

%matplotlib inline
import matplotlib; matplotlib.rcParams['figure.dpi'] = 100
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO)

from cache.alert_cache import AlertCache
from core.magellan_planning import (
    compute_merit, filter_observable_targets, write_magellan_catalog,
    radec_to_sexagesimal, INSTRUMENT_PRESETS,
)

cache = AlertCache(cache_dir=CACHE_DIR)
print(f'Cache initialized: {cache.db_path}')
print(f'Instrument: {INSTRUMENT} — {INSTRUMENT_PRESETS.get(INSTRUMENT, {}).get("description", "?")}')

In [ ]:
# Load peak-fit targets from SQLite cache
targets = cache.get_peak_fit_targets()

if targets is not None and len(targets) > 0:
    print(f'Loaded {len(targets)} targets from cache')
    print(f'  Cached at: {targets["cached_at"].iloc[0]}')

    # Summary statistics
    if 'ddf_field' in targets.columns:
        print(f'\nBy DDF field:')
        for field, cnt in targets['ddf_field'].value_counts().sort_index().items():
            if pd.notna(field) and field:
                print(f'  {field}: {cnt}')

    if 'peak_fit_status' in targets.columns:
        print(f'\nFit status:')
        for status, cnt in targets['peak_fit_status'].value_counts().items():
            print(f'  {status}: {cnt}')

    n_merit = targets['merit'].notna().sum()
    print(f'\nTargets with merit scores: {n_merit}')
    if n_merit > 0:
        print(f'  Merit range: {targets["merit"].min():.4f} to {targets["merit"].max():.4f}')
else:
    print('No targets in cache. Run supernova_monitor.ipynb through the peak fitting cells first.')
    targets = pd.DataFrame()

In [ ]:
# Recompute merit for the planned observing night
# (The cached merit used MJD at pipeline run time; this updates for the actual night)

if len(targets) > 0:
    obs_mjd = Time(OBS_DATE).mjd
    targets['delta_t'] = obs_mjd - targets['peak_mjd']
    targets['merit'] = compute_merit(
        targets['delta_t'], targets['peak_mag'],
        tau=MERIT_TAU,
        mag_optimal=MERIT_MAG_OPTIMAL,
        mag_bright=MERIT_MAG_BRIGHT,
        mag_faint=MERIT_MAG_FAINT,
    )

    print(f'Merit recomputed for observing night {OBS_DATE} (MJD {obs_mjd:.3f})')
    print(f'  tau = {MERIT_TAU}d, m_opt = {MERIT_MAG_OPTIMAL}')

    n_merit = targets['merit'].notna().sum()
    print(f'  {n_merit} targets with valid merit')
    if n_merit > 0:
        print(f'  Merit range: {targets["merit"].dropna().min():.4f} to {targets["merit"].dropna().max():.4f}')

    # Filter by fit status if requested
    if not INCLUDE_NO_FIT:
        before = len(targets)
        targets = targets[
            targets['peak_fit_status'].isin(['ok', 'underdetermined']) |
            (INCLUDE_NO_FIT)
        ].copy()
        print(f'  After fit-status filter: {len(targets)}/{before}')
else:
    print('No targets to process.')

In [ ]:
# Filter for targets observable from Las Campanas on the planned night

if len(targets) > 0:
    observable = filter_observable_targets(
        targets, OBS_DATE,
        max_airmass=MAX_AIRMASS,
        min_hours_up=MIN_HOURS_UP,
    )

    print(f'\nObservable targets: {len(observable)}/{len(targets)}')

    if len(observable) > 0:
        # Airmass chart
        fig, ax = plt.subplots(figsize=(12, 5))

        # Sort by RA for the chart
        obs_sorted = observable.sort_values('ra')

        scatter = ax.scatter(
            obs_sorted['ra'], obs_sorted['min_airmass'],
            c=obs_sorted['merit'], cmap='RdYlGn', vmin=0, vmax=1,
            s=40, edgecolors='black', linewidths=0.5,
        )
        plt.colorbar(scatter, label='Merit')
        ax.set_xlabel('RA (deg)')
        ax.set_ylabel('Minimum Airmass')
        ax.set_title(f'Observable Targets — {OBS_DATE} from Las Campanas')
        ax.set_ylim(0.9, MAX_AIRMASS + 0.2)
        ax.invert_yaxis()
        ax.axhline(y=MAX_AIRMASS, color='red', linestyle='--', alpha=0.5, label=f'airmass={MAX_AIRMASS}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Summary by DDF
        if 'ddf_field' in observable.columns:
            print('\nObservable by DDF:')
            for field, cnt in observable['ddf_field'].value_counts().sort_index().items():
                if pd.notna(field) and field:
                    print(f'  {field}: {cnt}')
else:
    observable = pd.DataFrame()
    print('No targets to filter.')

In [ ]:
# Select targets above merit threshold, sorted by RA

if len(observable) > 0:
    # Apply merit cutoff
    selected = observable[observable['merit'] >= MIN_MERIT].copy()
    selected = selected.sort_values('ra').reset_index(drop=True)

    print(f'Selected {len(selected)} targets (merit >= {MIN_MERIT})')

    if len(selected) > 0:
        # Build display table
        display_rows = []
        for i, (_, row) in enumerate(selected.iterrows()):
            ra_str, dec_str = radec_to_sexagesimal(row['ra'], row['dec'])
            display_rows.append({
                'Rank': i + 1,
                'Object ID': row.get('object_id', '?'),
                'RA': ra_str,
                'Dec': dec_str,
                'DDF': row.get('ddf_field', '?'),
                'Peak mag': f"{row['peak_mag']:.2f}" if pd.notna(row.get('peak_mag')) else '--',
                'dt (days)': f"{row['delta_t']:+.1f}" if pd.notna(row.get('delta_t')) else '--',
                'Merit': f"{row['merit']:.3f}" if pd.notna(row.get('merit')) else '--',
                'Airmass': f"{row['min_airmass']:.2f}" if pd.notna(row.get('min_airmass')) else '--',
                'Hours up': f"{row['hours_observable']:.1f}" if pd.notna(row.get('hours_observable')) else '--',
                'Status': row.get('peak_fit_status', '?'),
            })
        display_df = pd.DataFrame(display_rows)
        display(display_df.style.set_properties(
            **{'text-align': 'center', 'font-size': '11px'}
        ).set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'center'), ('font-size', '11px')]}
        ]))
    else:
        print(f'No targets above merit threshold {MIN_MERIT}.')
        print('Try lowering MIN_MERIT or adjusting merit parameters.')
else:
    selected = pd.DataFrame()
    print('No observable targets available.')

In [ ]:
# Write Magellan TCS catalog file

if len(selected) > 0:
    sigma_m = (MERIT_MAG_FAINT - MERIT_MAG_BRIGHT) / 4.0
    outfile = os.path.join(OUTPUT_DIR, f'magellan_targets_{OBS_DATE.replace("-", "")}.cat')

    write_magellan_catalog(
        selected, outfile,
        instrument=INSTRUMENT,
        obs_date=OBS_DATE,
        merit_params={
            'tau': MERIT_TAU,
            'mag_optimal': MERIT_MAG_OPTIMAL,
            'sigma_m': sigma_m,
        },
    )

    print(f'Wrote {len(selected)} targets to {outfile}')
    print()

    # Display first 30 lines
    with open(outfile) as f:
        for i, line in enumerate(f):
            if i >= 30:
                print(f'  ... ({len(selected)} targets total)')
                break
            print(line.rstrip())
else:
    print('No targets to write.')

In [ ]:
# Diagnostic plots: merit surface, and sky map

if len(observable) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # Panel 1: Merit vs delta_t
    ax = axes[0]
    valid = observable['merit'].notna()
    ax.scatter(observable.loc[valid, 'delta_t'], observable.loc[valid, 'merit'],
              c=observable.loc[valid, 'peak_mag'], cmap='viridis', s=20, alpha=0.7)
    ax.set_xlabel('Time from peak (days)')
    ax.set_ylabel('Merit')
    ax.set_title('Merit vs. Time from Peak')
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax.grid(True, alpha=0.3)

    # Panel 2: Merit vs peak_mag
    ax = axes[1]
    ax.scatter(observable.loc[valid, 'peak_mag'], observable.loc[valid, 'merit'],
              c=observable.loc[valid, 'delta_t'], cmap='coolwarm', s=20, alpha=0.7)
    ax.set_xlabel('Peak magnitude')
    ax.set_ylabel('Merit')
    ax.set_title('Merit vs. Peak Magnitude')
    ax.axvline(x=MERIT_MAG_OPTIMAL, color='gray', linestyle='--', alpha=0.5,
              label=f'm_opt={MERIT_MAG_OPTIMAL}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 3: Sky map
    ax = axes[2]
    sc = ax.scatter(observable['ra'], observable['dec'],
                   c=observable['merit'].fillna(0), cmap='RdYlGn',
                   vmin=0, vmax=1, s=25, edgecolors='black', linewidths=0.3)
    plt.colorbar(sc, ax=ax, label='Merit')
    ax.set_xlabel('RA (deg)')
    ax.set_ylabel('Dec (deg)')
    ax.set_title(f'Sky Map — {OBS_DATE}')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Merit surface visualization
    fig2, ax2 = plt.subplots(figsize=(8, 6))
    dt_grid = np.linspace(-30, 30, 200)
    mag_grid = np.linspace(17, 24, 200)
    DT, MAG = np.meshgrid(dt_grid, mag_grid)
    M = compute_merit(DT, MAG, tau=MERIT_TAU, mag_optimal=MERIT_MAG_OPTIMAL,
                      mag_bright=MERIT_MAG_BRIGHT, mag_faint=MERIT_MAG_FAINT)

    cs = ax2.contourf(DT, MAG, M, levels=20, cmap='RdYlGn')
    plt.colorbar(cs, label='Merit')
    ax2.set_xlabel('Time from peak (days)')
    ax2.set_ylabel('Peak magnitude')
    ax2.set_title('Merit Function Surface')
    ax2.invert_yaxis()  # Brighter at top

    # Overlay actual targets
    valid = observable['merit'].notna()
    ax2.scatter(observable.loc[valid, 'delta_t'], observable.loc[valid, 'peak_mag'],
              c='black', s=15, marker='x', alpha=0.8, label='Targets')
    ax2.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No targets for diagnostic plots.')